# ShelfTwin 07: temporal engine calibration and simulation

**CPU only: no GPU quota used** (*Accelerator: None*; Internet not needed). Uses all CPU cores in parallel.

1. Reproduces the comparison (single-frame, single-frame + mask, majority vote, ShelfTwin) on 20 simulated 8-hour runs (160 camera-hours).
2. Grid-searches `llr_empty`, `llr_stocked`, `act_threshold`, `decay_per_s` to minimise false alerts per camera-hour with recall ≥ 99 % and p95 delay ≤ 30 s.
   Tuning uses seeds 100-109; the result is reported on the untouched seeds 0-19. Finished configs are saved as they complete,
   so attaching this notebook's previous output resumes the search.
3. Scores an attached CSV of recorded outputs (`slot_id, ts, coverage, visibility, quality, needs_restock`) if present.

In [ ]:
SMOKE_TEST = False
import os, sys, shutil, glob
for d in ["/kaggle/working/st/shelftwin/engine", "/kaggle/working/st/sim", "/kaggle/working/outputs"]:
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, "/kaggle/working/st")
prev = glob.glob("/kaggle/input/**/engine_grid.jsonl", recursive=True)   # resume a previous search
if prev and not os.path.exists("/kaggle/working/outputs/engine_grid.jsonl"):
    shutil.copy(prev[0], "/kaggle/working/outputs/engine_grid.jsonl"); print("resuming from", prev[0])

In [ ]:
%%writefile /kaggle/working/st/shelftwin/__init__.py
"""ShelfTwin: trusted shelf-availability decisions from noisy camera detections."""
__version__ = "0.1.0"

In [ ]:
%%writefile /kaggle/working/st/shelftwin/engine/__init__.py
from .temporal import (EngineConfig, TemporalEngine, Decision, SlotTrack,
                       OK, WAIT, ACT, CLEAR, UNKNOWN, HEALTHY, EMPTY)
from .baselines import SingleFrameBaseline, MajorityVoteBaseline

__all__ = ["EngineConfig", "TemporalEngine", "Decision", "SlotTrack", "SingleFrameBaseline",
           "MajorityVoteBaseline", "OK", "WAIT", "ACT", "CLEAR", "UNKNOWN", "HEALTHY", "EMPTY"]

In [ ]:
%%writefile /kaggle/working/st/shelftwin/engine/temporal.py
"""Temporal state engine (spec Section 3.8).

Each slot is an occupancy-grid cell. Evidence L is the log-odds that the slot
needs restock:

    L_t = exp(-lambda * dt) * L_{t-1} + v_t * q_t * l(c_t)

A single confident frame never opens a task; hidden or low-quality views add no
evidence and, if they persist, the slot is reported as CANNOT SEE (UNKNOWN).
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field, asdict


# Slot states
HEALTHY = "HEALTHY"
EMPTY = "EMPTY"

# Engine decisions
OK = "OK"
WAIT = "WAIT"
ACT = "ACT"
CLEAR = "CLEAR"
UNKNOWN = "UNKNOWN"  # shown as CANNOT SEE in the UI


@dataclass
class EngineConfig:
    min_visibility: float = 0.6
    min_quality: float = 0.5
    empty_cov: float = 0.40
    full_cov: float = 0.45
    llr_empty: float = 1.6
    llr_stocked: float = 2.2
    decay_per_s: float = 0.02
    act_threshold: float = 4.0
    clear_threshold: float = -1.0
    unknown_after_s: float = 20.0
    score_min: float = -6.0
    score_max: float = 8.0

    @classmethod
    def from_dict(cls, d: dict | None) -> "EngineConfig":
        d = d or {}
        known = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}
        return cls(**known)


@dataclass
class SlotTrack:
    state: str = HEALTHY
    score: float = 0.0
    last_t: float | None = None
    last_clean_t: float | None = None
    last_coverage: float | None = None
    clean_views: int = 0          # clean confirming views since the last state change
    blocked_views: int = 0
    total_views: int = 0


@dataclass
class Decision:
    slot_id: str
    t: float
    decision: str
    state: str
    score: float
    coverage: float
    visibility: float
    quality: float
    weight: float
    clean_views: int
    last_clean_t: float | None

    def to_dict(self) -> dict:
        return asdict(self)


class TemporalEngine:
    """Visibility-weighted log-odds evidence with decay and hysteresis."""

    def __init__(self, config: EngineConfig | None = None):
        self.cfg = config or EngineConfig()
        self.tracks: dict[str, SlotTrack] = {}

    def track(self, slot_id: str) -> SlotTrack:
        if slot_id not in self.tracks:
            self.tracks[slot_id] = SlotTrack()
        return self.tracks[slot_id]

    def evidence(self, coverage: float) -> float:
        """l(c): positive means 'needs restock'."""
        if coverage < self.cfg.empty_cov:
            return self.cfg.llr_empty
        if coverage > self.cfg.full_cov:
            return -self.cfg.llr_stocked
        return 0.0

    def update(self, slot_id: str, t: float, coverage: float, visibility: float,
               quality: float = 1.0) -> str:
        return self.update_full(slot_id, t, coverage, visibility, quality).decision

    def update_full(self, slot_id: str, t: float, coverage: float, visibility: float,
                    quality: float = 1.0) -> Decision:
        cfg = self.cfg
        tr = self.track(slot_id)
        tr.total_views += 1

        if tr.last_t is not None:
            dt = max(0.0, t - tr.last_t)
            tr.score *= math.exp(-cfg.decay_per_s * dt)
        tr.last_t = t
        if tr.last_clean_t is None:
            tr.last_clean_t = t  # grace period starts at the first observation

        def out(decision: str, w: float) -> Decision:
            return Decision(slot_id, t, decision, tr.state, tr.score, coverage, visibility,
                            quality, w, tr.clean_views, tr.last_clean_t)

        if visibility < cfg.min_visibility or quality < cfg.min_quality:
            tr.blocked_views += 1
            if t - tr.last_clean_t >= cfg.unknown_after_s:
                return out(UNKNOWN, 0.0)
            return out(WAIT, 0.0)

        w = visibility * quality
        llr = self.evidence(coverage)
        tr.score = min(cfg.score_max, max(cfg.score_min, tr.score + w * llr))
        tr.last_clean_t = t
        tr.last_coverage = coverage
        if llr != 0.0:
            tr.clean_views += 1

        if tr.state == HEALTHY and tr.score >= cfg.act_threshold:
            tr.state = EMPTY
            tr.clean_views = 0
            return out(ACT, w)
        if tr.state == EMPTY and tr.score <= cfg.clear_threshold:
            tr.state = HEALTHY
            tr.clean_views = 0
            return out(CLEAR, w)
        if tr.state == HEALTHY and tr.score <= 0.0:
            return out(OK, w)
        return out(WAIT, w)

    def touch(self, slot_id: str, t: float, clean: bool) -> str:
        """Advance time for a slot whose view did not change (change gate said 'same scene').

        A repeated view is not an independent observation, so it adds NO evidence (otherwise one
        wrong frame would be re-counted every frame). It only applies decay and, if the cached
        view is clean, keeps the slot 'seen' so it does not drift into CANNOT SEE."""
        cfg = self.cfg
        tr = self.track(slot_id)
        if tr.last_t is not None:
            tr.score *= math.exp(-cfg.decay_per_s * max(0.0, t - tr.last_t))
        tr.last_t = t
        if tr.last_clean_t is None:
            tr.last_clean_t = t
        if clean:
            tr.last_clean_t = t
        elif t - tr.last_clean_t >= cfg.unknown_after_s:
            return UNKNOWN
        return OK if (tr.state == HEALTHY and tr.score <= 0.0) else WAIT

    def pending(self, slot_id: str, margin: float = 0.3) -> bool:
        """True while evidence is building toward a state change (needs fresh views)."""
        tr = self.tracks.get(slot_id)
        if tr is None:
            return False
        if tr.state == HEALTHY:
            return tr.score > margin
        return tr.score < 0.0  # EMPTY with stocked evidence arriving (a refill to verify)

    def reset(self, slot_id: str | None = None) -> None:
        if slot_id is None:
            self.tracks.clear()
        else:
            self.tracks.pop(slot_id, None)

    def snapshot(self) -> dict[str, dict]:
        return {k: asdict(v) for k, v in self.tracks.items()}

In [ ]:
%%writefile /kaggle/working/st/shelftwin/engine/baselines.py
"""Baselines the temporal engine is compared against (spec Section 3.8 / 7.1)."""
from __future__ import annotations

from collections import deque

from .temporal import ACT, CLEAR, OK, WAIT, HEALTHY, EMPTY


class SingleFrameBaseline:
    """Alerts on the first empty-looking frame."""

    def __init__(self, empty_cov: float = 0.40, use_mask: bool = False, min_visibility: float = 0.6):
        self.empty_cov = empty_cov
        self.use_mask = use_mask
        self.min_visibility = min_visibility
        self.state: dict[str, str] = {}

    def update(self, slot_id, t, coverage, visibility, quality=1.0) -> str:
        st = self.state.get(slot_id, HEALTHY)
        if self.use_mask and visibility < self.min_visibility:
            return WAIT
        looks_empty = coverage < self.empty_cov
        if st == HEALTHY and looks_empty:
            self.state[slot_id] = EMPTY
            return ACT
        if st == EMPTY and not looks_empty:
            self.state[slot_id] = HEALTHY
            return CLEAR
        return OK if st == HEALTHY else WAIT


class MajorityVoteBaseline:
    """Alerts when k of the last n usable frames look empty."""

    def __init__(self, n: int = 5, k: int = 3, use_mask: bool = True, empty_cov: float = 0.40,
                 min_visibility: float = 0.6):
        self.n, self.k = n, k
        self.use_mask = use_mask
        self.empty_cov = empty_cov
        self.min_visibility = min_visibility
        self.hist: dict[str, deque] = {}
        self.state: dict[str, str] = {}

    def update(self, slot_id, t, coverage, visibility, quality=1.0) -> str:
        if self.use_mask and visibility < self.min_visibility:
            return WAIT
        h = self.hist.setdefault(slot_id, deque(maxlen=self.n))
        h.append(coverage < self.empty_cov)
        st = self.state.get(slot_id, HEALTHY)
        votes = sum(h)
        if st == HEALTHY and votes >= self.k:
            self.state[slot_id] = EMPTY
            return ACT
        if st == EMPTY and (len(h) - votes) >= self.k:
            self.state[slot_id] = HEALTHY
            return CLEAR
        return OK if st == HEALTHY else WAIT

In [ ]:
%%writefile /kaggle/working/st/sim/simulate.py
"""Shelf simulator for the temporal engine (spec Section 7.1).

Generates a ground-truth world (units per slot, shopper occlusions, refills) and
noisy per-frame detector outputs, then replays the *same* observations through
every decision method so the comparison is fair.

Usage:
    python sim/simulate.py 20            # 20 runs (seeds 0-19), writes sim/results.json
    python sim/simulate.py 3 --quick     # 3 runs, 2 trading hours each
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from shelftwin.engine import (ACT, EngineConfig, MajorityVoteBaseline,  # noqa: E402
                              SingleFrameBaseline, TemporalEngine)


@dataclass
class SimConfig:
    n_slots: int = 24
    frame_dt_s: float = 2.0
    hours: float = 8.0
    units_min: int = 6
    units_max: int = 12
    demand_min_per_h: float = 0.5
    demand_max_per_h: float = 6.0
    blocks_per_h: float = 25.0
    block_logmean: float = 1.8
    block_logsd: float = 0.7
    block_min_s: float = 2.0
    block_max_s: float = 30.0
    p_gap_when_blocked: float = 0.70
    p_mask_miss: float = 0.05
    p_glare_gap: float = 0.01
    p_empty_misread: float = 0.03
    restock_at_units: int = 2
    refill_min_s: float = 600.0
    refill_max_s: float = 2400.0


@dataclass
class World:
    """Arrays shaped (n_slots, n_frames)."""
    t: np.ndarray
    needs: np.ndarray       # bool: slot needs restock (<= 2 units)
    coverage: np.ndarray    # detector coverage
    visibility: np.ndarray  # mask-reported visibility
    quality: np.ndarray
    causes: dict | None = None   # per-frame noise sources, for error analysis
    units: np.ndarray | None = None


def simulate_world(seed: int, cfg: SimConfig) -> World:
    rng = np.random.default_rng(seed)
    n_frames = int(cfg.hours * 3600 / cfg.frame_dt_s)
    S = cfg.n_slots
    t = np.arange(n_frames) * cfg.frame_dt_s

    cap = rng.integers(cfg.units_min, cfg.units_max + 1, size=S)
    rate = rng.uniform(cfg.demand_min_per_h, cfg.demand_max_per_h, size=S) / 3600.0

    units = np.zeros((S, n_frames), dtype=np.int32)
    for s in range(S):
        u = int(cap[s])
        refill_at = None
        lam = rate[s] * cfg.frame_dt_s
        takes = rng.poisson(lam, size=n_frames)
        for f in range(n_frames):
            if refill_at is not None and t[f] >= refill_at:
                u = int(cap[s])
                refill_at = None
            u = max(0, u - int(takes[f]))
            if u <= cfg.restock_at_units and refill_at is None:
                refill_at = t[f] + rng.uniform(cfg.refill_min_s, cfg.refill_max_s)
            units[s, f] = u
    needs = units <= cfg.restock_at_units

    # Shopper occlusion episodes (per slot Poisson process)
    blocked = np.zeros((S, n_frames), dtype=bool)
    mask_missed = np.zeros((S, n_frames), dtype=bool)
    horizon = cfg.hours * 3600
    for s in range(S):
        n_ev = rng.poisson(cfg.blocks_per_h * cfg.hours)
        starts = rng.uniform(0, horizon, size=n_ev)
        durs = np.clip(rng.lognormal(cfg.block_logmean, cfg.block_logsd, size=n_ev),
                       cfg.block_min_s, cfg.block_max_s)
        missed = rng.random(n_ev) < cfg.p_mask_miss
        for st, du, mi in zip(starts, durs, missed):
            f0 = int(np.ceil(st / cfg.frame_dt_s))
            f1 = int(np.floor((st + du) / cfg.frame_dt_s))
            if f1 < f0:
                continue
            blocked[s, f0:f1 + 1] = True
            if mi:
                mask_missed[s, f0:f1 + 1] = True

    # Detector coverage given true units
    r = rng.random((S, n_frames))
    cov = np.where(units >= 3, rng.uniform(0.55, 1.0, (S, n_frames)), 0.0)
    low = (units >= 1) & (units <= 2)
    cov = np.where(low, rng.uniform(0.05, 0.35, (S, n_frames)), cov)
    empty = units == 0
    misread = rng.random((S, n_frames)) < cfg.p_empty_misread
    cov = np.where(empty & ~misread, rng.uniform(0.0, 0.06, (S, n_frames)), cov)
    cov = np.where(empty & misread, rng.uniform(0.5, 0.9, (S, n_frames)), cov)
    glare = (units >= 3) & (r < cfg.p_glare_gap)
    cov = np.where(glare & ~blocked, rng.uniform(0.0, 0.2, (S, n_frames)), cov)

    # While blocked, the detector sees a gap 70% of the time
    blocked_gap = blocked & (rng.random((S, n_frames)) < cfg.p_gap_when_blocked)
    cov = np.where(blocked_gap, rng.uniform(0.0, 0.3, (S, n_frames)), cov)

    vis = np.where(blocked, rng.uniform(0.0, 0.5, (S, n_frames)), 1.0)
    vis = np.where(blocked & mask_missed, 1.0, vis)
    qual = np.ones((S, n_frames))
    causes = dict(blocked_mask_ok=blocked & ~mask_missed, blocked_mask_missed=blocked & mask_missed,
                  glare=glare & ~blocked, empty_misread=empty & misread)
    return World(t=t, needs=needs, coverage=cov, visibility=vis, quality=qual, causes=causes, units=units)


def run_method(make, world: World, return_alerts: bool = False):
    S, F = world.coverage.shape
    method = make()
    alerts = []  # (slot, frame)
    for s in range(S):
        sid = f"s{s}"
        cov, vis, q = world.coverage[s], world.visibility[s], world.quality[s]
        for f in range(F):
            if method.update(sid, float(world.t[f]), float(cov[f]), float(vis[f]), float(q[f])) == ACT:
                alerts.append((s, f))
    sc = score(alerts, world)
    return (sc, alerts) if return_alerts else sc


def episodes(needs_row: np.ndarray) -> list[tuple[int, int]]:
    d = np.diff(np.concatenate([[0], needs_row.astype(np.int8), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))  # [start, end)


def score(alerts, world: World) -> dict:
    S, F = world.coverage.shape
    correct = sum(1 for s, f in alerts if world.needs[s, f])
    false = len(alerts) - correct
    by_slot: dict[int, list[int]] = {}
    for s, f in alerts:
        by_slot.setdefault(s, []).append(f)
    n_events, detected, delays = 0, 0, []
    for s in range(S):
        fs = np.array(sorted(by_slot.get(s, [])), dtype=int)
        for a, b in episodes(world.needs[s]):
            n_events += 1
            hit = fs[(fs >= a) & (fs < b)]
            if hit.size:
                detected += 1
                delays.append(world.t[hit[0]] - world.t[a])
    cam_hours = F * (world.t[1] - world.t[0]) / 3600.0
    return dict(alerts=len(alerts), correct=correct, false=false, events=n_events,
                detected=detected, delays=delays, camera_hours=cam_hours)


def aggregate(parts: list[dict]) -> dict:
    alerts = sum(p["alerts"] for p in parts)
    correct = sum(p["correct"] for p in parts)
    false = sum(p["false"] for p in parts)
    events = sum(p["events"] for p in parts)
    detected = sum(p["detected"] for p in parts)
    delays = np.concatenate([np.asarray(p["delays"], dtype=float) for p in parts]) if parts else np.array([])
    hours = sum(p["camera_hours"] for p in parts)
    return dict(
        false_alerts_per_cam_hour=false / hours if hours else 0.0,
        precision=correct / alerts if alerts else 0.0,
        recall=detected / events if events else 0.0,
        median_delay_s=float(np.median(delays)) if delays.size else float("nan"),
        p95_delay_s=float(np.percentile(delays, 95)) if delays.size else float("nan"),
        alerts=alerts, false_alerts=false, true_events=events, camera_hours=hours,
    )


def methods(engine_cfg: EngineConfig | None = None):
    return {
        "Single-frame": lambda: SingleFrameBaseline(use_mask=False),
        "Single-frame + person mask": lambda: SingleFrameBaseline(use_mask=True),
        "Majority vote 3 of 5 + mask": lambda: MajorityVoteBaseline(n=5, k=3, use_mask=True),
        "ShelfTwin engine": lambda: TemporalEngine(engine_cfg or EngineConfig()),
    }


def run(n_runs: int = 20, sim_cfg: SimConfig | None = None, engine_cfg: EngineConfig | None = None,
        only: list[str] | None = None, verbose: bool = True) -> dict:
    sim_cfg = sim_cfg or SimConfig()
    ms = methods(engine_cfg)
    if only:
        ms = {k: v for k, v in ms.items() if k in only}
    parts: dict[str, list[dict]] = {k: [] for k in ms}
    t0 = time.time()
    for seed in range(n_runs):
        world = simulate_world(seed, sim_cfg)
        for name, make in ms.items():
            parts[name].append(run_method(make, world))
        if verbose:
            print(f"  run {seed + 1}/{n_runs} done ({time.time() - t0:.0f}s)", flush=True)
    return {name: aggregate(p) for name, p in parts.items()}


def print_table(results: dict) -> None:
    hdr = f"{'Method':<30} {'FA/cam-h':>9} {'Prec':>7} {'Recall':>7} {'Med(s)':>7} {'p95(s)':>7}"
    print(hdr)
    print("-" * len(hdr))
    for name, r in results.items():
        print(f"{name:<30} {r['false_alerts_per_cam_hour']:>9.2f} {r['precision'] * 100:>6.1f}% "
              f"{r['recall'] * 100:>6.1f}% {r['median_delay_s']:>7.0f} {r['p95_delay_s']:>7.0f}")
    any_r = next(iter(results.values()))
    print(f"\nTrue restock events: {any_r['true_events']}, camera-hours: {any_r['camera_hours']:.0f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="?", type=int, default=20)
    ap.add_argument("--quick", action="store_true", help="2 trading hours per run")
    ap.add_argument("--out", default=str(ROOT / "sim" / "results.json"))
    args = ap.parse_args()
    sim_cfg = SimConfig(hours=2.0) if args.quick else SimConfig()
    res = run(args.runs, sim_cfg)
    print_table(res)
    Path(args.out).write_text(json.dumps({"sim_config": asdict(sim_cfg), "n_runs": args.runs,
                                          "results": res}, indent=2))
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/st/calibrate_engine.py
"""Engine calibration (spec 6.8): grid-search llr_empty, llr_stocked, act_threshold, decay
to minimise false alerts per camera-hour subject to recall >= 99% and p95 delay <= 30 s.

Tuning uses simulator seeds 100-109; the chosen config is reported on held-out seeds 0-19,
so the reported numbers are not tuned on the data they are measured on. If a recorded CSV
(slot_id, ts, coverage, visibility, quality, needs_restock) is attached, it is evaluated too.
"""
from __future__ import annotations

import itertools
import os
from multiprocessing import Pool
from pathlib import Path

import numpy as np

from shelftwin.engine import EngineConfig, TemporalEngine
from sim.simulate import SimConfig, World, aggregate, run_method, simulate_world

GRID = dict(llr_empty=[1.2, 1.6, 2.0], llr_stocked=[1.6, 2.2, 2.8],
            act_threshold=[3.0, 3.5, 4.0, 4.5, 5.0], decay_per_s=[0.01, 0.02, 0.04])
TUNE_SEEDS = list(range(100, 110))
_WORLDS: list[World] = []


def _init(seeds, sim_cfg):
    global _WORLDS
    _WORLDS = [simulate_world(s, sim_cfg) for s in seeds]


def _eval(params: dict) -> dict:
    cfg = EngineConfig(**params)
    r = aggregate([run_method(lambda: TemporalEngine(cfg), w) for w in _WORLDS])
    return dict(params=params, **r)


def grid_search(sim_cfg: SimConfig | None = None, grid: dict | None = None, seeds=TUNE_SEEDS,
                min_recall=0.99, max_p95=30.0, workers: int | None = None,
                cache_path: Path | None = None) -> tuple[dict, list[dict]]:
    """Parallel over all CPU cores; with `cache_path` (JSON lines) finished configs are kept,
    so an interrupted search resumes where it stopped."""
    import json
    grid = grid or GRID
    sim_cfg = sim_cfg or SimConfig()
    combos = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]
    done: dict[str, dict] = {}
    if cache_path is not None and Path(cache_path).exists():
        for line in Path(cache_path).read_text().splitlines():
            try:
                r = json.loads(line)
                done[json.dumps(r["params"], sort_keys=True)] = r
            except Exception:
                continue
    todo = [c for c in combos if json.dumps(c, sort_keys=True) not in done]
    if done:
        print(f"Resuming grid search: {len(done)} configs cached, {len(todo)} to run")
    workers = workers or max(1, os.cpu_count() or 1)
    results = [done[json.dumps(c, sort_keys=True)] for c in combos if json.dumps(c, sort_keys=True) in done]
    if todo:
        with Pool(min(workers, len(todo)), initializer=_init, initargs=(seeds, sim_cfg)) as pool:
            for r in pool.imap_unordered(_eval, todo):
                results.append(r)
                if cache_path is not None:
                    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
                    with open(cache_path, "a") as f:
                        f.write(json.dumps(r, default=float) + "\n")
    ok = [r for r in results if r["recall"] >= min_recall and r["p95_delay_s"] <= max_p95]
    if not ok:
        print("No config met the constraints; relaxing to best recall.")
        ok = sorted(results, key=lambda r: -r["recall"])[:10]
    best = min(ok, key=lambda r: (r["false_alerts_per_cam_hour"], r["median_delay_s"]))
    return best, results


def evaluate_recorded_csv(path: Path, cfg: EngineConfig) -> dict:
    """Score the engine on recorded per-frame outputs with ground truth."""
    import pandas as pd
    df = pd.read_csv(path).sort_values(["slot_id", "ts"])
    alerts, correct, events, detected, delays = 0, 0, 0, 0, []
    for sid, g in df.groupby("slot_id"):
        e = TemporalEngine(cfg)
        need = g["needs_restock"].astype(bool).to_numpy()
        ts = g["ts"].to_numpy()
        acts = [i for i, r in enumerate(g.itertuples()) if
                e.update(sid, r.ts, r.coverage, r.visibility, getattr(r, "quality", 1.0)) == "ACT"]
        alerts += len(acts)
        correct += sum(need[i] for i in acts)
        d = np.diff(np.concatenate([[0], need.astype(int), [0]]))
        for a, b in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
            events += 1
            hit = [i for i in acts if a <= i < b]
            if hit:
                detected += 1
                delays.append(ts[hit[0]] - ts[a])
    hours = df.groupby("slot_id")["ts"].agg(lambda s: s.max() - s.min()).max() / 3600
    return dict(alerts=alerts, precision=correct / max(1, alerts), recall=detected / max(1, events),
                false_alerts_per_cam_hour=(alerts - correct) / max(hours, 1e-9),
                median_delay_s=float(np.median(delays)) if delays else None)


# ============================================================================ experiments / error analysis
SENSITIVITY = dict(
    p_gap_when_blocked=[0.3, 0.5, 0.7, 0.9],      # how often a shopper makes the detector see a gap
    p_mask_miss=[0.0, 0.05, 0.15, 0.30],          # person-mask failures
    p_glare_gap=[0.0, 0.01, 0.03, 0.06],          # glare/blur false gaps on stocked slots
    blocks_per_h=[10, 25, 50, 100],               # shopper traffic
    frame_dt_s=[1.0, 2.0, 4.0, 8.0],              # camera frame interval
)


def _sens_job(args):
    from dataclasses import replace
    from sim.simulate import methods
    param, value, base_cfg, seeds, engine_params = args
    cfg = replace(base_cfg, **{param: value})
    ms = methods(EngineConfig(**engine_params) if engine_params else None)
    keep = ("Majority vote 3 of 5 + mask", "ShelfTwin engine")
    parts = {k: [] for k in keep}
    for sd in seeds:
        w = simulate_world(sd, cfg)
        for k in keep:
            parts[k].append(run_method(ms[k], w))
    return dict(param=param, value=value, **{k: aggregate(v) for k, v in parts.items()})


def sensitivity(base_cfg: SimConfig | None = None, sweeps: dict | None = None, seeds=(0, 1, 2, 3),
                engine_params: dict | None = None, workers: int | None = None) -> list[dict]:
    """How false alerts / recall / delay degrade as each noise source gets worse (one at a time),
    for the engine and the strongest baseline. All (param, value) jobs run in parallel."""
    base_cfg = base_cfg or SimConfig(hours=4.0)
    sweeps = sweeps or SENSITIVITY
    jobs = [(p, v, base_cfg, list(seeds), engine_params) for p, vs in sweeps.items() for v in vs]
    with Pool(min(len(jobs), workers or os.cpu_count() or 1)) as pool:
        return pool.map(_sens_job, jobs)


def attribute_false_alerts(engine_params: dict | None = None, sim_cfg: SimConfig | None = None,
                           seeds=(0, 1, 2, 3), lookback_frames: int = 10) -> dict:
    """Error analysis: for every false ACT, which noise sources were present in the frames
    leading up to it? (A false alert can have several contributing causes.)"""
    from collections import Counter
    sim_cfg = sim_cfg or SimConfig()
    cfg = EngineConfig(**(engine_params or {}))
    causes, n_false, examples = Counter(), 0, []
    delays = []
    for sd in seeds:
        w = simulate_world(sd, sim_cfg)
        sc, alerts = run_method(lambda: TemporalEngine(cfg), w, return_alerts=True)
        delays += list(sc["delays"])
        for s, f in alerts:
            if w.needs[s, f]:
                continue
            n_false += 1
            f0 = max(0, f - lookback_frames)
            present = [k for k, arr in w.causes.items() if arr[s, f0:f + 1].any()]
            if not present:
                present = ["none of the modelled sources"]
            for k in present:
                causes[k] += 1
            if len(examples) < 20:
                examples.append(dict(seed=sd, slot=int(s), frame=int(f), causes=present,
                                     coverage=[round(float(x), 2) for x in w.coverage[s, f0:f + 1]],
                                     visibility=[round(float(x), 2) for x in w.visibility[s, f0:f + 1]],
                                     units=[int(x) for x in w.units[s, f0:f + 1]]))
    d = np.asarray(delays, float)
    return dict(n_false=n_false, cause_counts=dict(causes), examples=examples,
                delay_quantiles={q: float(np.percentile(d, q)) for q in (50, 90, 95, 99)} if d.size else {})

In [ ]:
import json, time
from pathlib import Path
from multiprocessing import Pool
from sim.simulate import SimConfig, run, print_table
from shelftwin.engine import EngineConfig
import calibrate_engine as ce
OUT = Path(os.environ.get("ST_WORK", "/kaggle/working")) / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
sim_cfg = SimConfig(hours=1.0) if SMOKE_TEST else SimConfig()
N_RUNS = 2 if SMOKE_TEST else 20
t0 = time.time()
base = run(N_RUNS, sim_cfg, verbose=False)
print_table(base); print(f"({time.time() - t0:.0f}s)")
grid = dict(llr_empty=[1.6], llr_stocked=[2.2], act_threshold=[3.5, 4.0], decay_per_s=[0.02]) if SMOKE_TEST else ce.GRID
t0 = time.time()
best, allres = ce.grid_search(sim_cfg, grid, seeds=ce.TUNE_SEEDS[:2] if SMOKE_TEST else ce.TUNE_SEEDS,
                              cache_path=OUT / ("engine_grid_smoke.jsonl" if SMOKE_TEST else "engine_grid.jsonl"))
print(f"{len(allres)} configs in {time.time() - t0:.0f}s on {os.cpu_count()} CPUs")
print("best on tuning seeds:", best["params"])
tuned = run(N_RUNS, sim_cfg, EngineConfig(**best["params"]), only=["ShelfTwin engine"], verbose=False)
print_table({"ShelfTwin (spec defaults)": base["ShelfTwin engine"], "ShelfTwin (tuned)": tuned["ShelfTwin engine"]})
rec = {}
for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) + glob.glob("/content/inputs/**/*.csv", recursive=True):
    try:
        import pandas as pd
        if {"slot_id", "ts", "coverage", "visibility", "needs_restock"} <= set(pd.read_csv(p, nrows=1).columns):
            rec[p] = dict(default=ce.evaluate_recorded_csv(Path(p), EngineConfig()),
                          tuned=ce.evaluate_recorded_csv(Path(p), EngineConfig(**best["params"])))
            print(p, rec[p])
    except Exception as e:
        print("skip", p, e)
import matplotlib.pyplot as plt
names = list(base) + ["ShelfTwin (tuned)"]
vals = [base[n]["false_alerts_per_cam_hour"] for n in base] + [tuned["ShelfTwin engine"]["false_alerts_per_cam_hour"]]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(names, vals); ax.set_xscale("log"); ax.set_xlabel("false alerts per camera-hour (log scale)")
for i, v in enumerate(vals): ax.text(v, i, f" {v:.2f}", va="center")
plt.tight_layout(); plt.savefig(OUT / "engine_false_alerts.png", dpi=120); plt.show()
json.dump(dict(baselines=base, tuned=tuned, best_params=best["params"], grid=grid, recorded=rec,
               sim_config=sim_cfg.__dict__, n_runs=N_RUNS), open(OUT / "engine_calibration.json", "w"), indent=2, default=str)
print("ALL DONE")